In [4]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
model=tf.keras.models.load_model("/content/drive/MyDrive/efficientnet_finetuned3.keras")
test_real = tf.data.Dataset.list_files( "/content/drive/MyDrive/face_detection_test/face_real/*.jpg",shuffle=False)
test_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",shuffle=False)
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img
test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)
def add_label(image, label):
  return image , label
test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))
test_dataset = test_real.concatenate(test_fake)
from tensorflow.keras.applications.efficientnet import preprocess_input
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label
test_dataset = test_dataset.map(preprocess)

In [6]:
BATCH_SIZE=32
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [7]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,397,544 (35.85 MB)

 Trainable params: 2,673,345 (10.20 MB)

 Non-trainable params: 1,377,507 (5.25 MB)

 Optimizer params: 5,346,692 (20.40 MB)

In [8]:
y_true = []
for _, labels in test_dataset:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

y_pred_prob = model.predict(test_dataset)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print(confusion_matrix(y_true,y_pred))

print(classification_report(y_true,y_pred,digits=4))

63/63 ━━━━━━━━━━━━━━━━━━━━ 36s 301ms/step
[[443 557]
 [285 715]]
              precision    recall  f1-score   support

           0     0.6085    0.4430    0.5127      1000
           1     0.5621    0.7150    0.6294      1000

    accuracy                         0.5790      2000
   macro avg     0.5853    0.5790    0.5711      2000
weighted avg     0.5853    0.5790    0.5711      2000

